# Pilot Runs — Colab Direct, No Google Drive

## What we are doing

Running the small-model pilot training from an ephemeral Colab runtime.

## Where to run

**Google Colab T4 GPU**

No Google Drive mount is used.

## You need to upload

1. `pilot_runs_scaffold_colab_direct.zip`
2. tokenizer artifact: `toolcall_spm_32k.model` or a zip containing it
3. 50M smoke shard zip containing `manifest.json`, `manifest_shards.jsonl`, and `shards/shard_*.bin`


## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Upload and extract the pilot scaffold

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile
import shutil

BASE_DIR = Path("/content/experiments")
PILOT_DIR = BASE_DIR / "pilot_runs"
TOKENIZER_BASE_DIR = BASE_DIR / "tokenizer"

BASE_DIR.mkdir(parents=True, exist_ok=True)

print("Upload pilot_runs_scaffold_colab_direct.zip")
uploaded = files.upload()

zip_candidates = [Path(name) for name in uploaded if name.endswith(".zip")]
if not zip_candidates:
    raise FileNotFoundError("No zip uploaded. Upload pilot_runs_scaffold_colab_direct.zip")

scaffold_zip = zip_candidates[0]
print("Using scaffold zip:", scaffold_zip)

extract_root = Path("/content/pilot_scaffold_extract")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(scaffold_zip, "r") as zf:
    zf.extractall(extract_root)

required = ["requirements.txt", "scripts", "configs"]
possible_roots = [extract_root] + [p for p in extract_root.iterdir() if p.is_dir()]
source_root = None

for candidate in possible_roots:
    if all((candidate / item).exists() for item in required):
        source_root = candidate
        break

if source_root is None:
    print("Extracted files:")
    for p in sorted(extract_root.rglob("*"))[:80]:
        print(" -", p)
    raise RuntimeError("Could not find scaffold root with requirements.txt, scripts/, configs/")

if PILOT_DIR.exists():
    shutil.rmtree(PILOT_DIR)
shutil.copytree(source_root, PILOT_DIR)

print("Pilot scaffold ready:", PILOT_DIR)
for p in sorted(PILOT_DIR.iterdir()):
    print(" -", p.name)


## 3. Install requirements

In [ ]:
%cd /content/experiments/pilot_runs
!pip -q install -r requirements.txt


## 4. Upload tokenizer artifact

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile
import shutil

TOKENIZER_ARTIFACT_DIR = Path("/content/experiments/tokenizer/artifacts/tokenizer")
TOKENIZER_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Upload toolcall_spm_32k.model directly OR a zip containing it")
uploaded = files.upload()

uploaded_paths = [Path(name) for name in uploaded]
extract_root = Path("/content/tokenizer_upload_extract")

if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

# Extract any uploaded zip files.
for path in uploaded_paths:
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path, "r") as zf:
            zf.extractall(extract_root)

# Copy direct uploads.
for path in uploaded_paths:
    if path.name == "toolcall_spm_32k.model":
        shutil.copy2(path, TOKENIZER_ARTIFACT_DIR / "toolcall_spm_32k.model")
    elif path.name == "toolcall_spm_32k.vocab":
        shutil.copy2(path, TOKENIZER_ARTIFACT_DIR / "toolcall_spm_32k.vocab")
    elif path.name == "tokenizer_manifest.json":
        shutil.copy2(path, TOKENIZER_ARTIFACT_DIR / "tokenizer_manifest.json")

# Copy from extracted zips if present.
model_matches = sorted(extract_root.rglob("toolcall_spm_32k.model"))
vocab_matches = sorted(extract_root.rglob("toolcall_spm_32k.vocab"))
manifest_matches = sorted(extract_root.rglob("tokenizer_manifest.json"))

if model_matches:
    shutil.copy2(model_matches[0], TOKENIZER_ARTIFACT_DIR / "toolcall_spm_32k.model")
if vocab_matches:
    shutil.copy2(vocab_matches[0], TOKENIZER_ARTIFACT_DIR / "toolcall_spm_32k.vocab")
if manifest_matches:
    shutil.copy2(manifest_matches[0], TOKENIZER_ARTIFACT_DIR / "tokenizer_manifest.json")

MODEL_PATH = TOKENIZER_ARTIFACT_DIR / "toolcall_spm_32k.model"
if not MODEL_PATH.exists():
    raise FileNotFoundError("Could not find/copy toolcall_spm_32k.model")

print("Tokenizer ready:", MODEL_PATH)
print("Files:")
for p in sorted(TOKENIZER_ARTIFACT_DIR.iterdir()):
    print(" -", p.name, round(p.stat().st_size / 1024**2, 3), "MB")


## 5. Upload and extract the 50M smoke shard zip

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile
import shutil

SMOKE_DIR = Path("/content/experiments/tokenizer/data/tokenized/smoke_50m")
if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR)
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

print("Upload smoke_50m_token_shards.zip or any zip containing manifest.json + shards/shard_*.bin")
uploaded = files.upload()

zip_candidates = [Path(name) for name in uploaded if name.endswith(".zip")]
if not zip_candidates:
    raise FileNotFoundError("No shard zip uploaded.")

shard_zip = zip_candidates[0]
print("Using shard zip:", shard_zip)

extract_root = Path("/content/smoke_50m_extract")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(shard_zip, "r") as zf:
    zf.extractall(extract_root)

# Detect the folder that contains shards/.
candidate_roots = [extract_root] + [p for p in extract_root.rglob("*") if p.is_dir()]
source_root = None

for candidate in candidate_roots:
    if (candidate / "shards").exists() and list((candidate / "shards").glob("shard_*.bin")):
        source_root = candidate
        break

# Some zips may contain shard_*.bin directly.
if source_root is None:
    direct_shards = list(extract_root.rglob("shard_*.bin"))
    if direct_shards:
        source_root = extract_root
        (source_root / "shards").mkdir(exist_ok=True)
        for shard in direct_shards:
            if shard.parent != source_root / "shards":
                shutil.copy2(shard, source_root / "shards" / shard.name)

if source_root is None:
    print("Extracted files:")
    for p in sorted(extract_root.rglob("*"))[:100]:
        print(" -", p)
    raise RuntimeError("Could not find shards/shard_*.bin in uploaded zip.")

# Copy source root contents into SMOKE_DIR.
if (source_root / "manifest.json").exists():
    shutil.copy2(source_root / "manifest.json", SMOKE_DIR / "manifest.json")

if (source_root / "manifest_shards.jsonl").exists():
    shutil.copy2(source_root / "manifest_shards.jsonl", SMOKE_DIR / "manifest_shards.jsonl")

shards_out = SMOKE_DIR / "shards"
shards_out.mkdir(parents=True, exist_ok=True)

for shard in sorted((source_root / "shards").glob("shard_*.bin")):
    shutil.copy2(shard, shards_out / shard.name)

print("Smoke shard ready:", SMOKE_DIR)
print("Shard count:", len(list(shards_out.glob("shard_*.bin"))))
print("Total shard size MB:", round(sum(p.stat().st_size for p in shards_out.glob("shard_*.bin")) / 1024**2, 2))
for p in sorted(SMOKE_DIR.iterdir()):
    print(" -", p.name)


## 6. Inspect pilot inputs

In [ ]:
%cd /content/experiments/pilot_runs

!python scripts/inspect_pilot_inputs.py \
  --data-dir ../tokenizer/data/tokenized/smoke_50m \
  --tokenizer ../tokenizer/artifacts/tokenizer/toolcall_spm_32k.model \
  --sample-tokens 256


## 7. Quick 100-step pilot

Run this first. It is only to catch runtime errors quickly.

In [ ]:
%cd /content/experiments/pilot_runs

!python scripts/train_pilot_lm.py \
  --config configs/tiny_smoke_50m_quick_100_steps.json


## 8. Full 1000-step tiny smoke pilot

Run this only after the 100-step pilot succeeds.

In [ ]:
%cd /content/experiments/pilot_runs

!python scripts/train_pilot_lm.py \
  --config configs/tiny_smoke_50m.json


## 9. Summarize and download result

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

%cd /content/experiments/pilot_runs

!python scripts/summarize_pilot_run.py \
  --run-dir runs/tiny_smoke_50m \
  --out results/tiny_smoke_50m_result.md

print(Path("results/tiny_smoke_50m_result.md").read_text())

archive_path = shutil.make_archive(
    "/content/pilot_tiny_smoke_results",
    "zip",
    root_dir="/content/experiments/pilot_runs",
    base_dir="results",
)

print("Downloading:", archive_path)
files.download(archive_path)
